[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bsheese/125/blob/08-rebuild/08_data_vis/08_6_Change_Over_Time.ipynb)

# 08.6: Change Over Time

A **line plot** connects points in order, which suits a question about how something changes over time. It is easy to choose between lines and bars by habit, so this notebook starts with how to make that choice. It then covers the issues that come up with lines: how many to draw, what the axis does to the slope, which time range to show and what a line hides.

This notebook opens in Google Colab. If Colab warns that the notebook was not authored by Google, click **Run anyway**. Your changes are not kept when you close the tab unless you choose **File > Save a copy in Drive**, so do that if you want to keep your work.

## Why a line needs an order

A line says that one point leads to the next and that the values in between make sense. That is true for time. It is not true for categories with no natural order. Here is the mean body mass of penguins on three islands, drawn as a line.

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

penguins = pd.read_csv("https://raw.githubusercontent.com/bsheese/125/08-rebuild/data/data_penguins.csv")
island_mass = penguins.groupby("island")["body_mass_g"].mean().reset_index()
print(island_mass)
ax = sns.lineplot(data=island_mass, x="island", y="body_mass_g")
ax.set_title("A line through categories")
ax.set_xlabel("Island")
ax.set_ylabel("Mean body mass (grams)")
plt.show()

The line appears to show mass falling as you move from Biscoe, at about 4716 grams, to Dream, at about 3713, and then leveling off at Torgersen, at about 3706. But there is nothing between Biscoe and Dream, and the islands are in this order only because it is alphabetical. In a different order the same numbers would make a different shape. These are three separate categories, so a bar chart fits, as you saw in notebook 08.2.

In [ ]:
ax = sns.barplot(data=island_mass.sort_values("body_mass_g", ascending=False), x="island", y="body_mass_g")
ax.set_title("Mean body mass by island")
ax.set_xlabel("Island")
ax.set_ylabel("Mean body mass (grams)")
plt.show()

The bars compare the islands without suggesting any path between them.

## Lines and bars over time

With time on the x axis, both charts can be reasonable, and the choice depends on what you want a reader to see. We use the airline data from notebook 08.1, where one row is one month. `parse_dates=` tells pandas to treat the `date` column as dates.

In [ ]:
flights = pd.read_csv("https://raw.githubusercontent.com/bsheese/125/08-rebuild/data/data_flights.csv", parse_dates=["date"])
ax = sns.lineplot(data=flights, x="date", y="passengers")
ax.set_title("Monthly airline passengers, 1949-1960")
ax.set_xlabel("Month")
ax.set_ylabel("Passengers (thousands)")
plt.show()

There are 144 months, far too many for separate bars, and the line shows two patterns at once: passengers rise steadily over the years, and they rise and fall within every year, reaching a peak of 622 thousand in July 1960. Now here are the totals for each year, which gives only 12 values.

In [ ]:
yearly = flights.groupby("year")["passengers"].sum().reset_index()
print(yearly.tail(3))
ax = sns.barplot(data=yearly, x="year", y="passengers")
ax.set_title("Total airline passengers by year")
ax.set_xlabel("Year")
ax.set_ylabel("Passengers (thousands)")
plt.show()

Each bar is a separate total, and its height from zero can be compared with the others: 1960, at 5714 thousand, is well over three times 1949, at 1520 thousand. A line would also be fine here. Bars put the emphasis on the separate amounts, and a line puts it on the trend.

Here is a summary of the choice:

| Your x axis | Chart | Why |
|---|---|---|
| Categories with no order | Bar | A line would suggest a path and values in between that do not exist |
| Time, with many points | Line | Too many bars to read, and the trend is usually the point |
| Time, with a few points, where each is a separate amount such as a yearly total | Bar or line | Bars stress the amounts and lines stress the trend |
| Ordered numbers such as a dose or a size | Line | The order is real, and values in between make sense |

## More than one line

`hue=` draws one line per value of a column, with a legend, when all of the lines come from one table. Here one row is one country in one year, and `Life_Expectancy` is in years.

In [ ]:
health = pd.read_csv("https://raw.githubusercontent.com/bsheese/125/08-rebuild/data/data_healthexp.csv")
ax = sns.lineplot(data=health, x="Year", y="Life_Expectancy", hue="Country")
ax.set_title("Life expectancy, 1970-2020")
ax.set_xlabel("Year")
ax.set_ylabel("Life expectancy (years)")
plt.show()
print(health[health["Year"] == 2020][["Country", "Life_Expectancy"]])

All six countries rise over the period. Japan ends highest at 84.7 years in 2020 and the USA ends lowest at 77.0. Most of the lines dip in 2020, and the USA falls the most, from 78.8 to 77.0. Six lines are about as many as you can follow by color.

If the lines come from separate tables, make two calls and give each one a `label=`. Both calls draw on the same chart, and `ax.legend()` adds the legend. Here are Illinois and the country as a whole.

In [ ]:
crime = pd.read_csv("https://raw.githubusercontent.com/bsheese/CSDS125ExampleData/master/data_corgis_state_crime.csv")
crime = crime.rename(columns={"Rates.Violent.All": "Violent_Rate"})
illinois = crime[crime["State"] == "Illinois"]
nation = crime[crime["State"] == "United States"]
ax = sns.lineplot(data=illinois, x="Year", y="Violent_Rate", label="Illinois")
sns.lineplot(data=nation, x="Year", y="Violent_Rate", label="United States")
ax.set_title("Violent crime rates, 1960-2012")
ax.set_xlabel("Year")
ax.set_ylabel("Violent crimes per 100,000 people")
ax.legend()
plt.show()

Both lines rise and then fall, and Illinois stays above the national rate in all 53 years. The gap was 281.0 at the Illinois peak in 1991 and only 27.9 by 2012.

A legend makes the reader look back and forth between the lines and the labels. With only a few lines, you can label them directly with `ax.text()`, which takes an x position, a y position and the text. `legend=False` turns off the legend.

In [ ]:
three = crime[crime["State"].isin(["Illinois", "Indiana", "Iowa"])]
ax = sns.lineplot(data=three, x="Year", y="Violent_Rate", hue="State", legend=False)
ax.set_title("Violent crime rates in three states, 1960-2012")
ax.set_xlabel("Year")
ax.set_ylabel("Violent crimes per 100,000 people")
ax.text(2013, 414.8, "Illinois")
ax.text(2013, 345.7, "Indiana")
ax.text(2013, 263.9, "Iowa")
ax.set_xlim(1960, 2022)
plt.show()

The reader can now see which line is which without leaving the lines. All three rise and then fall, but not at the same time: Illinois peaks at 1039.2 in 1991, Iowa at 354.4 in 1995 and Indiana at 537.0 in 1996. The positions are the 2012 values, typed in by hand, so this approach suits a small number of lines.

## Too many lines

Here is every state, with one color each.

In [ ]:
states = crime[crime["State"] != "United States"]
ax = sns.lineplot(data=states, x="Year", y="Violent_Rate", hue="State", legend=False)
ax.set_title("Violent crime rates, every state")
ax.set_xlabel("Year")
ax.set_ylabel("Violent crimes per 100,000 people")
plt.show()

With 51 lines you can see the overall rise and fall and one line far above the others, which is the District of Columbia, but you cannot follow or identify any other line. This kind of chart is sometimes called a spaghetti chart. There are two good fixes. One is to choose only the lines your question needs, as we did with three states. The other is to give each line its own small chart. `sns.relplot()` with `kind="line"` and `col=` does this, using the same axes in each one.

In [ ]:
grid = sns.relplot(data=health, x="Year", y="Life_Expectancy", col="Country", col_wrap=3, kind="line", height=2.5)
grid.set_axis_labels("Year", "Life expectancy (years)")
grid.set_titles("{col_name}")
plt.show()

Each country is easy to follow now, and because the axes match, you can still compare them, such as the USA's lower ending point and its drop at the end compared with Japan.

## A surprise: a band instead of lines

If there are several rows for each value of `x` and you do not separate them with `hue=`, `sns.lineplot()` does not draw several lines. It draws the mean at each `x`, with a shaded band showing the uncertainty around it. Sometimes that is what you want, but it can also surprise you if you forgot `hue=`.

In [ ]:
ax = sns.lineplot(data=health, x="Year", y="Life_Expectancy")
ax.set_title("Mean life expectancy across six countries")
ax.set_xlabel("Year")
ax.set_ylabel("Life expectancy (years)")
plt.show()

This line averages the six countries equally in each year, so it is not life expectancy for any actual place. If you see a band that you did not ask for, check whether you have more than one row per `x`.

## The axis and the slope

Notebook 08.2 said that bars must start at zero, because their length stands for the amount. A line is read by its position and direction, so it can use a narrower range, and this is often necessary. Here is life expectancy in the USA on two axes.

In [ ]:
usa = health[health["Country"] == "USA"]
ax = sns.lineplot(data=usa, x="Year", y="Life_Expectancy")
ax.set_title("Life expectancy in the USA, axis from 0")
ax.set_xlabel("Year")
ax.set_ylabel("Life expectancy (years)")
ax.set_ylim(0, 90)
plt.show()

ax = sns.lineplot(data=usa, x="Year", y="Life_Expectancy")
ax.set_title("Life expectancy in the USA, seaborn's default axis")
ax.set_xlabel("Year")
ax.set_ylabel("Life expectancy (years)")
plt.show()
print(usa[usa["Year"].isin([1970, 2019, 2020])][["Year", "Life_Expectancy"]])

With an axis from 0, the line is nearly flat, and the rise from 70.9 years in 1970 and the drop from 78.8 in 2019 to 77.0 in 2020 are hard to see. Since nobody lives to an age near 0 on average, all that space tells us nothing. The narrower axis shows the changes clearly. A narrower axis also makes every change look steeper, however, and people who study charts disagree about how much that matters. So treat this as a judgment call. A line does not have to start at zero, but you should choose a range that does not exaggerate a change, and when you read a chart, check the numbers on the axis before you react to the slope.

## Which time range?

The time range you show can change the story more than anything else. Here are the airline passengers again, for just a few months.

In [ ]:
late_1960 = flights[(flights["date"] >= "1960-07-01") & (flights["date"] <= "1960-11-01")]
ax = sns.lineplot(data=late_1960, x="date", y="passengers")
ax.set_title("Airline passengers, July to November 1960")
ax.set_xlabel("Month")
ax.set_ylabel("Passengers (thousands)")
ax.tick_params(axis="x", rotation=45)
plt.show()

Passengers fall from 622 thousand to 390 thousand, a drop of about 37 percent in four months, which looks like a collapse. But the chart of all 144 months showed that this drop happens every autumn, within a steady long-term rise. Each number here is accurate, and the chart still misleads. When you choose a range, choose it because of your question, name it in the title, and ask what the chart would show if it started earlier or ended later. It is also worth checking whether the last period in a chart is complete, since a partial year or month can look like a sudden drop. `ax.tick_params()` tilts the dates here, because such long labels do not fit side by side.

## What a line can hide

A line connects whatever points exist, so it can hide missing data. `marker="o"` draws a dot at each actual data point.

In [ ]:
france = health[health["Country"] == "France"]
print(france["Year"].head(6).to_list())
ax = sns.lineplot(data=france, x="Year", y="Life_Expectancy", marker="o")
ax.set_title("Life expectancy in France")
ax.set_xlabel("Year")
ax.set_ylabel("Life expectancy (years)")
plt.show()

The first data points are 1970, 1975, 1980, 1985 and then 1990, and only after that are there yearly values. Without the dots you could not tell that the early part of the line is drawn across gaps of five years. Showing the points is a good idea when a series has few points or uneven gaps.

## Reusing a chart with a function

If you are making the same chart for several states, a function saves you from repeating yourself, as functions did in module 04.

In [ ]:
def plot_state(state_name):
    rows = crime[crime["State"] == state_name]
    ax = sns.lineplot(data=rows, x="Year", y="Violent_Rate")
    ax.set_title("Violent crime rate in " + state_name)
    ax.set_xlabel("Year")
    ax.set_ylabel("Violent crimes per 100,000 people")
    plt.show()

plot_state("Vermont")

Vermont's rate starts at 9.5 in 1960, peaks at 184.2 in 1979 and is 142.6 by 2012. Its peak comes about 12 years before the one in Illinois, which is the kind of difference you can spot quickly when a chart for another state takes only one more line of code.

## Practice problems

These problems are not collected or graded. They are here so that you can find out whether you can write this code yourself before the reading check and the pair activity. Write your answer in the cell under each problem, run it, and then run the check cell below it, which tells you whether your answer is right. The check cells are collapsed because they contain the answers, so try the problem before you open one.

Run the next two cells first. The first one creates fresh data for the problems, and the second sets up the checks.

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

flights = pd.read_csv("https://raw.githubusercontent.com/bsheese/125/08-rebuild/data/data_flights.csv", parse_dates=["date"])
health = pd.read_csv("https://raw.githubusercontent.com/bsheese/125/08-rebuild/data/data_healthexp.csv")
crime = pd.read_csv("https://raw.githubusercontent.com/bsheese/CSDS125ExampleData/master/data_corgis_state_crime.csv")
crime = crime.rename(columns={"Rates.Violent.All": "Violent_Rate", "Rates.Property.All": "Property_Rate"})
illinois_rows = crime[crime["State"] == "Illinois"]
indiana_rows = crime[crime["State"] == "Indiana"]
france_rows = health[health["Country"] == "France"]
usa_rows = health[health["Country"] == "USA"]

In [ ]:
#@title Run this cell to set up the checks
import numpy as np
import pandas as pd


def same(got, want):
    """True when got matches want: same labels and values for a series, same value for a number."""
    try:
        if isinstance(want, pd.DataFrame):
            if not isinstance(got, pd.DataFrame) or list(got.columns) != list(want.columns) or len(got) != len(want):
                return False
            if not pd.api.types.is_integer_dtype(want.index) and list(got.index) != list(want.index):
                return False
            got, want = got.reset_index(drop=True), want.reset_index(drop=True)
            for column in want.columns:
                if pd.api.types.is_numeric_dtype(want[column]):
                    if not np.allclose(pd.to_numeric(got[column], errors="coerce").astype(float), want[column].astype(float), equal_nan=True):
                        return False
                elif got[column].fillna("").astype(str).to_list() != want[column].fillna("").astype(str).to_list():
                    return False
            return True
        if isinstance(want, pd.Series):
            if not isinstance(got, pd.Series) or list(got.index) != list(want.index):
                return False
            if pd.api.types.is_numeric_dtype(want):
                return bool(np.allclose(got.astype(float), want.astype(float), equal_nan=True))
            return list(got) == list(want)
        if isinstance(want, (list, np.ndarray)):
            return list(got) == list(want)
        if isinstance(want, str):
            return got == want
        if isinstance(got, (pd.Series, pd.DataFrame, list, np.ndarray)):
            return False
        return bool(np.isclose(float(got), float(want)))
    except (TypeError, ValueError):
        return False


def check(number, name, want, hint, unchanged=None):
    """Print feedback for one problem. unchanged is (variable name, its starting value)."""
    if name not in globals():
        print(f"Problem {number}: not yet. No variable named {name} exists. Did you run your answer cell?")
    elif not same(globals()[name], want):
        print(f"Problem {number}: not yet. {hint}")
    elif unchanged is not None and not globals()[unchanged[0]].equals(unchanged[1]):
        print(f"Problem {number}: {name} is right, but {unchanged[0]} changed. Work on a copy, then run the setup cell again.")
    else:
        print(f"Problem {number}: correct.")


def check_chart(number, name, test, hint, y_label=True):
    """Check a chart stored in a variable: it has labels, and test(chart) is True."""
    if name not in globals():
        print(f"Problem {number}: not yet. No variable named {name} exists. Did you run your answer cell?")
        return
    chart = globals()[name]
    if not hasattr(chart, "get_title"):
        print(f"Problem {number}: not yet. Store what the seaborn function returns in {name}.")
    elif not chart.get_title() or not chart.get_xlabel() or (y_label and not chart.get_ylabel()):
        print(f"Problem {number}: not yet. Give the chart a title and label its axes.")
    else:
        try:
            ok = bool(test(chart))
        except (AttributeError, IndexError, TypeError, ValueError):
            ok = False
        print(f"Problem {number}: correct." if ok else f"Problem {number}: not yet. {hint}")

### Problem 1
You want to compare mean ticket sales across five movie genres, which have no natural order. Should you use a line or a bar chart? Store your answer in a variable named `chart_1`, as one of `"line"` or `"bar"`.

In [ ]:
# Problem 1

In [ ]:
#@title Check Problem 1
check(1, "chart_1", "bar", "A line would suggest a path between genres.")

### Problem 2
You have 120 months of unemployment rates and want to show the trend. Should you use a line or a bar chart? Store your answer in a variable named `chart_2`, as one of `"line"` or `"bar"`.

In [ ]:
# Problem 2

In [ ]:
#@title Check Problem 2
check(2, "chart_2", "line", "120 bars would be hard to read, and the trend is the point.")

### Problem 3
Make a line plot of `Spending_USD` by `Year` for all six countries, using one `sns.lineplot()` call with `hue=`. Store it in `spending_lines`. Give it a title and label both axes.

In [ ]:
# Problem 3

In [ ]:
#@title Check Problem 3
check_chart(3, "spending_lines", lambda c: {t.get_text() for t in c.get_legend().get_texts()} == set(health["Country"]), "Use sns.lineplot() with data=health and hue=\"Country\".")

### Problem 4
Draw two lines of `Property_Rate` by `Year` on one chart, one from `illinois_rows` and one from `indiana_rows`, with a legend. Store the chart in `two_lines`. Give it a title and label both axes.

In [ ]:
# Problem 4

In [ ]:
#@title Check Problem 4
check_chart(4, "two_lines", lambda c: len(c.get_lines()) >= 2 and c.get_legend() is not None, "Make two lineplot calls, give each a label=, and call .legend().")

### Problem 5
The cell below already has code that draws Indiana's violent crime rate. Run it and look at the chart. The rate really did rise from 323.3 in 2010 to 345.7 in 2012, but the chart shows only those three years and its title makes them sound like the whole story. Fix the code so that it uses all of `indiana_rows`, and change the title so that it names the years shown, 1960 and 2012, instead of making a claim. Keep the chart stored in `full_line`.

In [ ]:
# Problem 5
recent = indiana_rows[indiana_rows["Year"] >= 2010]
full_line = sns.lineplot(data=recent, x="Year", y="Violent_Rate")
full_line.set_title("Violent crime is climbing fast in Indiana")
full_line.set_xlabel("Year")
full_line.set_ylabel("Violent crimes per 100,000 people")
plt.show()

In [ ]:
#@title Check Problem 5
check_chart(5, "full_line", lambda c: len(c.get_lines()[0].get_xdata()) == 53 and "1960" in c.get_title() and "2012" in c.get_title(), "Use data=indiana_rows with no year filter, and put 1960 and 2012 in the title.")

### Problem 6
Make a line plot of `Spending_USD` by `Year` using `france_rows`, with a dot at each data point, and store it in `france_line`. Give it a title and label both axes.

In [ ]:
# Problem 6

In [ ]:
#@title Check Problem 6
check_chart(6, "france_line", lambda c: c.get_lines()[0].get_marker() == "o" and len(c.get_lines()[0].get_xdata()) == len(france_rows), "Use sns.lineplot() with data=france_rows and marker=\"o\".")

### Problem 7
You draw `sns.lineplot(data=health, x="Year", y="Spending_USD")` and see a single line with a shaded band. What is the most likely reason? Store your answer in a variable named `band_cause`, as one of `"several rows per x"` or `"missing values"`.

In [ ]:
# Problem 7

In [ ]:
#@title Check Problem 7
check(7, "band_cause", "several rows per x", "This table has one row per country in each year.")

## What's next

You can now choose between lines and bars and handle the common issues with lines. The notebooks so far have each taken one kind of question. Notebook 08.7, the last in this module, brings them together: given a chart, how do you decide whether to trust it, and how do you fix it if you should not?